# EcoOrbit — reproducible observed-data PoC

This reviewer notebook uses bundled **real processed Tanager site spectra**, observed heat/air products, and source-linked plume metadata. It requires no credentials or 3.3 GB imagery download. Raw-scene acquisition and correction details are in `README.md` and `docs/methodology.md`. The 48-site industrial-versus-school proxy test is small; results do not attribute emissions to individual sites.


In [1]:
from pathlib import Path
import os, sys, json, pandas as pd
root = Path.cwd()
if root.name == "notebooks": root = root.parent
assert (root / "data/training_sites.csv").exists(), "Start Jupyter in the project root or notebooks directory"
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: EcoOrbit_v2


## 1. Reproduce the hyperspectral site test

Tanager L2A provider-corrected reflectance, 57 binned spectral bands and three indices, 36 training and 12 held-out sites balanced by city and proxy label. PCA components are selected with training-only cross-validation. Rebuilding this test uses the included measured feature rows, not the raw 3.3 GB HDF5 scenes.


In [2]:
from src.evaluate_sites import run as evaluate
evaluate()
m = json.loads(Path("results/real/classifier/all_city_test_metrics.json").read_text())
print(f"Headline: {m['test_accuracy']:.1%} accuracy on {m['n_test']} held-out sites; AUC {m['test_auc']:.3f}")


{
  "protocol": "Site-level 75/25 split stratified by city and label; fixed seed 813; train-only three-fold model selection",
  "n_train": 36,
  "n_test": 12,
  "test_by_city_and_class": "{('Detroit', 0): 2, ('Detroit', 1): 2, ('Jacksonville', 0): 2, ('Jacksonville', 1): 2, ('Rochester', 0): 2, ('Rochester', 1): 2}",
  "n_industrial_sites": 24,
  "n_school_controls": 24,
  "negative_categories": {
    "OSM school comparison": 24
  },
  "feature_count": 60,
  "candidate_components_cv_accuracy": {
    "4": 0.6944444444444443,
    "8": 0.7222222222222222,
    "12": 0.7222222222222222
  },
  "selected_pca_components": 8,
  "test_accuracy": 0.9166666666666666,
  "test_auc": 0.9444444444444444,
  "test_precision": 0.8571428571428571,
  "test_recall": 1.0,
  "test_confusion_matrix_school_industrial": [
    [
      5,
      1
    ],
    [
      0,
      6
    ]
  ],
  "interpretation": "Registry and OSM polygon labels are proxies; small held-out site set; score is not factory-wide accuracy or 

![All-city held-out classifier accuracy](../results/real/classifier/all_city_test_metrics.png)

The two labels are industrial registry points and OSM school comparison points. Green-space patches provide additional spectral context, not independent test labels.


## 2. Validate real published CH₄/CO₂ plumes

These are three **published source-linked examples**, not matches at the proposed Arizona, Algeria or Jeddah sites. Spot kg/h rates cannot become annual facility totals without persistence and activity evidence.


In [3]:
from src.import_methane_plumes import published
plumes = published()
print(plumes[["site_name","gas","scene_timestamp","reported_rate_kg_h","source_url"]].to_string(index=False))


Validated 3 source-linked published CH4/CO2 plume examples; target-case attribution not implied
                          site_name gas      scene_timestamp  reported_rate_kg_h                                                                                                          source_url
Carbon Mapper product-guide example CH4 2024-04-20T10:14:48Z         3610.581227                                                                     https://carbonmapper.org/articles/product-guide
                Seropédica landfill CH4 2024-09-29T13:16:28Z         2836.000000 https://carbonmapper.org/articles/carbon-mapper-data-from-tanager1-satellite-reveals-methane-and-co2-super-emitters
   Singrauli electricity generation CO2 2024-11-01T05:19:27Z       838000.000000 https://carbonmapper.org/articles/carbon-mapper-data-from-tanager1-satellite-reveals-methane-and-co2-super-emitters


## 3. Heat, NO₂/SO₂, wind and vegetation context

Landsat temperatures are surface values; Sentinel-5P is a regional vertical column. Missing satellite coverage is retained as missing, never zero.


In [4]:
heat = pd.read_csv("results/real/heat/site_heat.csv")
gas = pd.read_csv("results/real/pollutants/site_no2_so2.csv")
wind = pd.read_csv("results/real/wind_by_city.csv")
print("Landsat rows:", len(heat), "distinct sites:", heat.facility_id.nunique())
print("Median surface excess °C by city:\n", heat.groupby("city").surface_excess_c.median().round(2).to_string())
print("Sentinel-5P status by city/product:\n", gas.groupby(["city","product","status"]).size().to_string())
print("ERA5 wind city-hour rows:", len(wind))
print("Green reference summary:", json.loads(Path("results/real/green_reference_summary.json").read_text())["comparison_scope"])


Landsat rows: 38 distinct sites: 19
Median surface excess °C by city:
 city
Detroit         0.66
Jacksonville    1.45
Rochester       4.00
Sentinel-5P status by city/product:
 city          product  status                                    
Detroit       no2      coarse column pixel; no source attribution    8
              so2      coarse column pixel; no source attribution    8
Jacksonville  no2      download pending                              8
              so2      download pending                              8
Rochester     no2      no catalog item in image-week window          8
              so2      no catalog item in image-week window          8
ERA5 wind city-hour rows: 504
Green reference summary: Green cluster-wide median versus 8 factory-neighborhood medians; green patches are spatial reference locations, not 12 independent NDVI measurements or verified parks.


![Observed Landsat heat](../results/real/heat/site_heat.png)

![Observed green comparison](../results/real/green_reference_comparison.png)


## 4. Build the offline website and figures

Open `EcoOrbit_Results_Website.html` in a browser. It embeds these output tables and figures, and its map shows the US pilot locations.


In [5]:
from src.build_website import build
website = build()
print("Website:", website.name, "—", round(website.stat().st_size/1e6, 2), "MB")


Website: EcoOrbit_Results_Website.html — 1.52 MB


## Optional source refresh

The original Tanager HDF5 scenes (about 3.3 GB total) can be fetched with `python -m src.multisite_download` and re-extracted using `python -m src.train_classifier`. Sentinel/Landsat refresh commands and exact scene IDs are in the README. Carbon Mapper case matching needs an exported CSV/GeoJSON and `python -m src.import_methane_plumes path/to/export.csv`; an empty match set is a legitimate outcome. Do not enter credentials into this notebook.
